In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("taxi") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 04:24:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark

In [7]:
spark.version

'3.5.1'

In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-11 04:24:28--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.64.222.158, 18.64.222.96, 18.64.222.103, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.64.222.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   122KB/s    in 5m 49s  

2026-03-11 04:30:19 (199 KB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
dfy = spark.read.parquet("/opt/spark-data/yellow_tripdata_2025-11.parquet")

In [5]:
dfy.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [10]:
dfy.repartition(4).write.mode("overwrite").parquet("/opt/spark-data/yellow_2025_11_repartitioned")

In [11]:
import os

files = [f for f in os.listdir("/opt/spark-data/yellow_2025_11_repartitioned") if f.endswith(".parquet")]
files

['part-00000-fed9ffdc-63bc-4299-8230-d79a99f82b65-c000.snappy.parquet',
 'part-00001-fed9ffdc-63bc-4299-8230-d79a99f82b65-c000.snappy.parquet',
 'part-00002-fed9ffdc-63bc-4299-8230-d79a99f82b65-c000.snappy.parquet',
 'part-00003-fed9ffdc-63bc-4299-8230-d79a99f82b65-c000.snappy.parquet']

In [12]:
sizes = []

for f in files:
    size = os.path.getsize(f"/opt/spark-data/yellow_2025_11_repartitioned/{f}") / (1024*1024)
    sizes.append(size)

sum(sizes) / len(sizes)

24.409069776535034

In [13]:
df = spark.read.parquet("/opt/spark-data/yellow_2025_11_repartitioned")

In [15]:
from pyspark.sql.functions import to_date, col

nov_15 = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15")
nov_15.count()

162604

In [17]:
from pyspark.sql.functions import expr

df.select(
    expr("max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))")
).alias("Longest hour").show()

[Stage 21:====================================>                     (5 + 3) / 8]

+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                   90|
+---------------------------------------------------------------------+



In [18]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-11 05:10:56--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.64.222.121, 18.64.222.96, 18.64.222.158, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.64.222.121|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.07s   

2026-03-11 05:10:57 (183 KB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [22]:
taxi_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load("/opt/spark-data/taxi_zone_lookup.csv")

In [23]:
taxi_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [24]:
joined_df = df.join(
    taxi_df,
    df.PULocationID == taxi_df.LocationID,
    "inner"
)

In [25]:
from pyspark.sql.functions import count

In [26]:
zone_counts = joined_df.groupBy("Zone") \
    .agg(count("*").alias("pickup_count"))

In [27]:
zone_counts.orderBy("pickup_count").show(1, False)

[Stage 30:====================================>                     (5 + 3) / 8]

+-------------+------------+
|Zone         |pickup_count|
+-------------+------------+
|Arden Heights|1           |
+-------------+------------+
only showing top 1 row

